# Procesar Parquet con Python y PySpark

Usaremos el mismo Parquet y el mismo cálculo en ambos casos. Antes de empezar, completa la [guía de entorno](../01_environment_pyspark.md). Este cuaderno genera sus propios datos.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from time import perf_counter

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("Comparar Parquet").getOrCreate()
carpeta = Path(mkdtemp(prefix="pyspark_parquet_"))
print("Archivos de la práctica:", carpeta)

## 1. Mirar una tabla pequeña

Un CSV es texto y se puede abrir con un editor. Parquet guarda los datos por columnas en formato binario, junto con sus tipos. Spark escribe una carpeta con archivos de datos dentro, tanto para CSV como para Parquet.

In [ ]:
pequeno = spark.createDataFrame(
    [(1, "Pan", 2), (2, "Manzanas", 4), (3, "Pan", 1)],
    ["pedido", "producto", "unidades"],
)
csv_pequeno = carpeta / "pequeno_csv"
parquet_pequeno = carpeta / "pequeno_parquet"
pequeno.coalesce(1).write.csv(str(csv_pequeno), header=True)
pequeno.coalesce(1).write.parquet(str(parquet_pequeno))

print(next(csv_pequeno.glob("part-*.csv")).read_text(encoding="utf-8"))
spark.read.parquet(str(parquet_pequeno)).printSchema()

## 2. Guardar la misma tabla grande en los dos formatos

Empezamos con un millón de filas sintéticas. Si la VM tiene poco espacio, prueba con 100 000. Si quieres aumentar el volumen, cambia la cifra y vuelve a ejecutar desde esta sección. Se crean dos copias de los datos y puede tardar. El tamaño se calcula sumando los archivos de datos de cada carpeta.

In [ ]:
filas = 1_000_000
grande = spark.range(filas, numPartitions=4).select(
    F.col("id").alias("pedido"),
    F.concat(F.lit("zona_"), (F.col("id") % 20)).alias("zona"),
    (F.col("id") % 12 + 1).alias("unidades"),
    F.sha2(F.col("id").cast("string"), 256).alias("referencia"),
)

csv_grande = carpeta / f"pedidos_{filas}_csv"
parquet_grande = carpeta / f"pedidos_{filas}_parquet"
grande.write.mode("overwrite").csv(str(csv_grande), header=True)
grande.write.mode("overwrite").parquet(str(parquet_grande))

for ruta, extension in [(csv_grande, "csv"), (parquet_grande, "parquet")]:
    bytes_datos = sum(f.stat().st_size for f in ruta.glob(f"part-*.{extension}"))
    print(f"{ruta.name}: {bytes_datos / 1024**2:.1f} MiB")

El CSV grande se genera para comparar cuánto ocupan los mismos datos en ambos formatos. La consulta que mediremos a continuación se hace sobre Parquet en los dos casos.

In [ ]:
def consulta(tabla):
    return tabla.filter(F.col("zona") == "zona_3").agg(F.sum("unidades"))

def medir(nombre, calcular):
    inicio = perf_counter()
    total = calcular()
    print(f"{nombre}: {total} unidades en {perf_counter() - inicio:.2f} s")
    return total



## 3. Ejercicio A · El mismo Parquet con Python y PySpark

Calcula el total de unidades de la zona 3 leyendo exactamente los mismos archivos Parquet. Usa PyArrow para leer solo `zona` y `unidades` por lotes y suma con Python. Después, haz la misma consulta con PySpark. Compara el resultado y los tiempos. Si falta PyArrow, instala `%pip install pyarrow` en el kernel y vuelve a ejecutar desde el inicio si necesitas reiniciarlo.

Antes de mirar la solución, intenta completar estas tres operaciones: abrir el Parquet, seleccionar la zona y sumar las unidades.

In [ ]:
# Escribe aquí tu solución.

### Solución del ejercicio A

El bucle de Python procesa un lote a la vez. PyArrow se encarga de leer Parquet. En PySpark expresamos el filtro y la suma con las operaciones del DataFrame.

In [ ]:
import pyarrow.parquet as pq

def sumar_parquet_con_python(ruta):
    total = 0
    for parte in ruta.glob("part-*.parquet"):
        with pq.ParquetFile(parte) as archivo:
            for lote in archivo.iter_batches(columns=["zona", "unidades"]):
                zonas, unidades = (columna.to_pylist() for columna in lote.columns)
                total += sum(valor for zona, valor in zip(zonas, unidades) if zona == "zona_3")
    return total

total_arrow = medir("Python + PyArrow con Parquet", lambda: sumar_parquet_con_python(parquet_grande))
total_spark = medir("PySpark con Parquet", lambda: consulta(
    spark.read.parquet(str(parquet_grande))
).first()[0])

assert total_arrow == total_spark 
consulta(spark.read.parquet(str(parquet_grande))).explain()

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: Spark y HDFS](07_spark_hdfs.ipynb)